# Agent Skill

预计阅读时间：10-15 分钟。

上一节的 Agent 已经具备 `bash` 工具和基本循环。本节只新增一个能力：**Skill 加载**。

为了避免重复，本节不重新实现上一节已经讲过的客户端、`run_bash` 和 `agent_loop`，而是直接从 [`03_agent.py`](03_agent.py) 中导入。

## 为什么需要 Skill

如果把所有规则都写进 `SYSTEM` prompt，提示词会越来越长，模型每次调用都要读大量无关内容。

Skill 的做法分两层：

| 层次 | 内容 | 什么时候进入上下文 |
| --- | --- | --- |
| Skill catalog | skill 名称和一句话简介 | 启动时放入 `SYSTEM` |
| Skill content | 完整 `SKILL.md` | 模型调用 `load_skill` 时才加载 |

本节示例使用 `python-docstring` skill：当任务涉及补充 Python docstring 时，Agent 可以先加载这个 skill，再按其中规则完成任务。

## 目录结构

本节使用如下目录：

```text
books/06_Agent/
├── 03_agent.py
└── skills/
    └── python-docstring/
        └── SKILL.md
```

`SKILL.md` 中保存了 skill 的名称、简介和完整规则。

## 1. 导入公共函数和 Skill 模块

`03_agent.py` 是在 `02_agent.py` 的基础上增加 Skill 机制。为了让关系更清楚，本节只直接复用 `02_agent.py` 和 `03_agent.py` 中共有且实现一致的基础函数。

这里直接复用的公共函数是：

- `run_bash`：执行 Bash 命令。

Skill 相关函数不再单独导入，而是通过 `skill_agent.xxx` 调用。这样可以清楚看到哪些能力来自 `03_agent.py` 新增的 Skill 机制。


In [1]:
import importlib
import json
import sys
from pathlib import Path

AGENT_DIR = Path("books/06_Agent") if Path("books/06_Agent").exists() else Path.cwd()
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))

base_agent = importlib.import_module("02_agent")
skill_agent = importlib.import_module("03_agent")

# 只直接复用 02_agent.py 和 03_agent.py 共有且实现一致的基础函数。
run_bash = base_agent.run_bash

print("Agent dir:", AGENT_DIR)
print("Skills dir:", skill_agent.SKILLS_DIR)


Agent dir: /Users/sunq/suibe_courses/modern_programming/books/06_Agent
Skills dir: /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skills


## 2. 查看 python-docstring skill

先读取当前示例 skill。它的作用是要求 Agent 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。

In [2]:
skill_path = skill_agent.SKILLS_DIR / "python-docstring" / "SKILL.md"
print(skill_path)
print(skill_path.read_text(encoding="utf-8"))

/Users/sunq/suibe_courses/modern_programming/books/06_Agent/skills/python-docstring/SKILL.md
---
name: python-docstring
description: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。
---

# Python Docstring Skill

当任务涉及“补充注释”“解释函数”“完善文档字符串”时，使用本 skill。

## 要求

- 使用 Google 风格 docstring；
- docstring 放在函数定义下一行；
- 包含 `Args`、`Returns`，必要时包含 `Raises`；
- 不改变函数原有逻辑；
- 修改后运行 `python3 -m py_compile 文件名` 检查语法。

## 示例

```python
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The sum of `a` and `b`.
    """
    return a + b
```



## 3. 解析 Skill frontmatter

`SKILL.md` 开头使用类似 YAML 的 frontmatter 保存元信息：

```markdown
---
name: python-docstring
description: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。
---
```

`03_agent.py` 中的 `parse_skill_frontmatter()` 会解析这段元信息。

In [3]:
raw_skill = skill_path.read_text(encoding="utf-8")
meta, body = skill_agent.parse_skill_frontmatter(raw_skill)
meta

{'name': 'python-docstring',
 'description': '为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。'}

## 4. 扫描 skills 目录

`scan_skills()` 会扫描 `skills/*/SKILL.md`，生成 skill registry。这个 registry 用来做两件事：

- 给 `SYSTEM` prompt 注入轻量 catalog；
- 让 `load_skill(name)` 可以安全读取指定 skill。

In [4]:
skill_registry = skill_agent.scan_skills()
skill_registry.keys()

dict_keys(['python-docstring'])

## 5. Skill catalog 和 load_skill

`list_skills()` 只返回名称和简介，适合放入 `SYSTEM`。

`load_skill(name)` 返回完整 `SKILL.md`，作为工具结果返回给模型。

In [5]:
print(skill_agent.list_skills())
print("\n--- full skill preview ---")
print(skill_agent.load_skill("python-docstring")[:500])

- python-docstring: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。

--- full skill preview ---
---
name: python-docstring
description: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。
---

# Python Docstring Skill

当任务涉及“补充注释”“解释函数”“完善文档字符串”时，使用本 skill。

## 要求

- 使用 Google 风格 docstring；
- docstring 放在函数定义下一行；
- 包含 `Args`、`Returns`，必要时包含 `Raises`；
- 不改变函数原有逻辑；
- 修改后运行 `python3 -m py_compile 文件名` 检查语法。

## 示例

```python
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The sum of `


## 6. SYSTEM 中的 Skill catalog

和上一节相比，`SYSTEM` 中多了一段 `Skills available`。它告诉模型当前有哪些 skill，但不立即放入完整规则。

In [6]:
print(skill_agent.SYSTEM)

You are a coding agent at /Users/sunq/suibe_courses/modern_programming/books/06_Agent. Use bash to solve tasks. Act, don't explain.

Skills available:
- python-docstring: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。

Use load_skill(name) to load full skill instructions when a task matches a skill.


## 7. 工具列表

本节只比上一节多一个工具：`load_skill`。`bash` 仍然保留，Agent 可以在加载 skill 后继续用 Bash 修改文件和运行检查。

In [7]:
[(tool["name"], tool["description"]) for tool in skill_agent.TOOLS]

[('bash', 'Run a shell command.'),
 ('load_skill', 'Load full instructions for a skill by name.')]

## 8. 工具执行体验

下面直接调用 `load_skill`，观察它返回的完整内容。真实 Agent 运行时，模型会通过 `tool_use` 调用这个工具。

In [8]:
print(skill_agent.load_skill("python-docstring"))

---
name: python-docstring
description: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。
---

# Python Docstring Skill

当任务涉及“补充注释”“解释函数”“完善文档字符串”时，使用本 skill。

## 要求

- 使用 Google 风格 docstring；
- docstring 放在函数定义下一行；
- 包含 `Args`、`Returns`，必要时包含 `Raises`；
- 不改变函数原有逻辑；
- 修改后运行 `python3 -m py_compile 文件名` 检查语法。

## 示例

```python
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The sum of `a` and `b`.
    """
    return a + b
```



## 9. 示例任务：补充 docstring

先创建一个没有 docstring 的 Python 文件。然后让 Agent 使用 skill 为函数补充 Google 风格 docstring，并运行语法检查。

In [9]:
demo_file = AGENT_DIR / "skill_demo.py"
demo_file.write_text(
    "def add(a: int, b: int) -> int:\n"
    "    return a + b\n",
    encoding="utf-8",
)
print(demo_file.read_text(encoding="utf-8"))

def add(a: int, b: int) -> int:
    return a + b



In [10]:
messages = [{
    "role": "user",
    "content": "请使用 python-docstring skill，为 skill_demo.py 中的函数补充 Google 风格 docstring，并运行 py_compile 检查语法。",
}]
skill_agent.agent_loop(messages)

response_content = messages[-1]["content"]
if isinstance(response_content, list):
    for block in response_content:
        if hasattr(block, "text"):
            print(block.text)

load_skill(python-docstring)
---
name: python-docstring
description: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。
---

# Python Docstring Skill

当任务涉及“补充注释”“解释函数”“完善文档字符串”时，使用本 skill。

## 要求

- 使用 Google 风格 docstring；
- docstring
$ cat /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skill_demo.py
def add(a: int, b: int) -> int:
    return a + b
$ cat > /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skill_demo.py << 'EOF'
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: The first integer.
        b: The second integer.

    Returns:
        The sum of `a` and `b`.
    """
    return a + b
EOF
(no output)
$ python3 -m py_compile /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skill_demo.py && echo "语法检查通过 ✅"
语法检查通过 ✅
$ cat /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skill_demo.py
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: The first integer.
  

## 10. 保存并查看历史

完整 `messages` 可以看到 skill 加载过程：

1. 用户请求补充 docstring；
2. 模型调用 `load_skill("python-docstring")`；
3. Agent 返回完整 `SKILL.md`；
4. 模型根据 skill 生成 Bash 命令修改文件；
5. Agent 执行命令并返回结果；
6. 模型运行语法检查；
7. 最后返回总结。

In [11]:
skill_agent.HISTORY_PATH.write_text(
    json.dumps(messages, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(f"History saved to {skill_agent.HISTORY_PATH}")
print(json.dumps(messages, ensure_ascii=False, indent=2, default=str)[:3000])

History saved to /Users/sunq/suibe_courses/modern_programming/books/06_Agent/03_history.json
[
  {
    "role": "user",
    "content": "请使用 python-docstring skill，为 skill_demo.py 中的函数补充 Google 风格 docstring，并运行 py_compile 检查语法。"
  },
  {
    "role": "assistant",
    "content": [
      "ToolUseBlock(id='call_4eeb6b02d5914037bde22ca9', caller=None, input={'name': 'python-docstring'}, name='load_skill', type='tool_use')",
      "ToolUseBlock(id='call_b13149f5d1614f139584be68', caller=None, input={'command': 'cat /Users/sunq/suibe_courses/modern_programming/books/06_Agent/skill_demo.py'}, name='bash', type='tool_use')"
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "tool_use_id": "call_4eeb6b02d5914037bde22ca9",
        "content": "---\nname: python-docstring\ndescription: 为 Python 函数补充 Google 风格 docstring，并保持代码逻辑不变。\n---\n\n# Python Docstring Skill\n\n当任务涉及“补充注释”“解释函数”“完善文档字符串”时，使用本 skill。\n\n## 要求\n\n- 使用 Google 风格 docstring；\n- docstring

## 小结

Skill 的意义在于：让 Agent 按需加载专项规则。

本节的关键点：

- 02 中已经存在的客户端、`bash` 工具和 `agent_loop` 可以复用；
- `SYSTEM` 中只放 skill 名称和简介；
- 完整规则保存在 `SKILL.md`；
- 模型需要时调用 `load_skill`；
- Agent loop 不需要变化，只是多了一个工具。